In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from lightgbm import LGBMClassifier
import shap

c:\Users\OTENE\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ====================== LOAD DATA & PREPROCESS ======================
df = pd.read_csv("cleaned_dataset.csv")

# Map target to numeric (handle common labels)
df["loan_status"] = df["loan_status"].map({"Y": 1, "N": 0, "y": 1, "n": 0}).astype(float)

# Clean and convert columns to numeric where needed
df["dependents"] = df["dependents"].replace("3+", 3)
df["dependents"] = pd.to_numeric(df["dependents"], errors="coerce").fillna(0).astype(int)

df["loan_amount"] = pd.to_numeric(df["loan_amount"], errors="coerce")
df["loan_amount"].fillna(df["loan_amount"].median(), inplace=True)

df["applicant_income"] = pd.to_numeric(df["applicant_income"], errors="coerce").fillna(0)
df["co_applicant_income"] = pd.to_numeric(df["co_applicant_income"], errors="coerce").fillna(0)
df["loan_amount_term"] = pd.to_numeric(df["loan_amount_term"], errors="coerce").fillna(df["loan_amount_term"].median())
df["credit_history"] = pd.to_numeric(df["credit_history"], errors="coerce").fillna(0)

# Map categorical string columns to numeric
df["gender"] = df["gender"].map({"Male": 1, "Female": 0}).fillna(0).astype(int)
df["married"] = df["married"].map({"Yes": 1, "No": 0}).fillna(0).astype(int)
df["education"] = df["education"].map({"Graduate": 0, "Not Graduate": 1}).fillna(0).astype(int)
df["self_employed"] = df["self_employed"].map({"Yes": 1, "No": 0}).fillna(0).astype(int)
df["property_area"] = df["property_area"].map({"Rural": 0, "Semiurban": 1, "Urban": 2}).fillna(1).astype(int)

# Derived features used by the model
df["total_income"] = df["applicant_income"] + df["co_applicant_income"]
df["income_loan_ratio"] = df["total_income"] / df["loan_amount"].replace({0: np.nan}).fillna(1)
df["total_income_log"] = np.log1p(df["total_income"])
df["loan_amount_log"] = np.log1p(df["loan_amount"])

# Define FEATURES to match model input
FEATURES = [
    "gender", "married", "dependents", "education", "self_employed",
    "applicant_income", "co_applicant_income", "loan_amount",
    "loan_amount_term", "credit_history", "property_area",
    "total_income", "income_loan_ratio", "total_income_log", "loan_amount_log",
]

# Drop rows where target is missing
df = df.dropna(subset=["loan_status"]) 

X = df[FEATURES]
y = df["loan_status"].astype(int)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)


Training shape: (491, 11)
Test shape: (123, 11)


In [9]:
# ====================== BEST MODEL (LightGBM) ======================
model = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model.fit(X_train, y_train)

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: gender: str, married: str, dependents: str, education: str, self_employed: str, property_area: str

In [ ]:
# Save model
joblib.dump(model, "best_model.joblib")
print("Model saved as best_model.joblib")



In [ ]:
# ====================== EVALUATION ======================
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n=== MODEL PERFORMANCE ===")
print(f"Accuracy       : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision      : {precision_score(y_test, y_pred):.4f}")
print(f"Recall         : {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score       : {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC        : {roc_auc_score(y_test, y_prob):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



In [ ]:
# ====================== RISK SCORING FUNCTION ======================
def get_risk_score(prob):
    return round((1 - prob) * 100, 1)

print("\n Group 2 Complete - Model Ready (No SMOTE)")